# Design of Buffer Tanks

This notebook implements the results and recommendations from the following paper:

- Faanes and Skogestad (2000). A systematic approach to the design of buffer tanks

## Model

Consider a buffer tank with liquid volume $V$, inlet flow rate
$q_{\text{in}}$, and outlet flow rate $q$.

The inlet and outlet quality variables (e.g. concentration or temperature) are
denoted by $c_{\text{in}}$ and $c$, respectively.

For a perfectly mixed tank, a component (or simplified energy) balance yields

$$
\frac{d(Vc)}{dt} = q_{\text{in}}\,c_{\text{in}} - q\,c
$$

In addition, the total mass balance (assuming constant density) is

$$
\frac{dV}{dt} = q_{\text{in}} - q
$$

**Symbols**

- $V$ — liquid volume in the tank $[ \mathrm{m}^3 ]$
- $q_{\text{in}}$ — inlet volumetric flow rate $[ \mathrm{m}^3\,\mathrm{s}^{-1} ]$
- $q$ — outlet volumetric flow rate $[ \mathrm{m}^3\,\mathrm{s}^{-1} ]$
- $c_{\text{in}}$ — inlet flow quality (concentration or temperature)  
  $[ \mathrm{kg}\,\mathrm{m}^{-3} \text{ or } \mathrm{K} ]$
- $c$ — outlet flow quality  
  $[ \mathrm{kg}\,\mathrm{m}^{-3} \text{ or } \mathrm{K} ]$
- $t$ — time $[ \mathrm{s} ]$


## Smoothing Quality Disturbances

### Transfer function for a single buffer tank

For quality disturbances, the objective of the buffer tank is to smoothen the quality response, $c(s) = h(s) q_{in}(s)$, so that the variations in $c$ are smaller than in $c_{in}$.

By linearizing at the steady-state operating point we get the following function for the quality of the outlet flow.

$$
c(s)=\frac{1}{\frac{V^*}{q^*} s+1}\left[c_{\text {in }}(s)+\frac{c^*_{\text {in }}-c^*}{q^*} q_{\text {in }}(s)\right]
$$

where $^*$ denotes the nominal (steady state) values.

In the case where $c^*_{\text {in}} = c^*$, the transfer function reduces to

$$
h(s) = \frac{1}{\tau_h s + 1}
$$

where $\tau_h = V^* / q^*$ is called the residence time (steady state).

### Transfer function for buffer tanks in series

$$
h(s) = \frac{1}{\big( \frac{\tau_h}{n} s + 1 \big)^n}
$$

### Frequency Response

**Magnitude**

$$
\left| H(j\omega) \right|
=
\frac{1}{
\left[
1 + \left( \omega \frac{\tau_h}{n} \right)^2
\right]^{\tfrac{n}{2}}
}
$$

**Phase**

$$
\angle H(j\omega)
=
-\,n \tan^{-1}\!\left(
\omega \frac{\tau_h}{n}
\right)
$$


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import sympy
import control as con

from bounded_random_walk import sample_bounded_random_walk

In [ ]:
plot_dir = Path("./plots")
plot_dir.mkdir(exist_ok=True)

FIGSIZE = (5.5, 5)

In [ ]:
def calc_amplitude_response(omega, tau_h, n):
    """Calculate the amplitude ratio of n buffer tanks in series.

    Arguments
    ----------
    omega : float
        Angular frequency (rad/min).
    tau_h : float
        Total residence time of system (min).
    n : int
        Number of buffer tanks in series.
    """
    return 1 / (1 + (omega * tau_h / n) ** 2) ** (n / 2)


def calc_phase_response(omega, tau_h, n):
    """Calculate the phase response of n buffer tanks in series.

    Arguments
    ----------
    omega : float
        Angular frequency (rad/min).
    tau_h : float
        Total residence time of system (min).
    n : int
        Number of buffer tanks in series.
    """
    return -n * np.arctan(omega * tau_h)

In [ ]:
# Input disturbance parameters
period = 10  # period of inlet disturbance (min)
omega = 2 * np.pi / period  # angular frequency (rad/min)
freq_times_tau_h = np.logspace(
    -1, 2, 101
)  # total volume of buffer tanks (m^3)

amplitude_responses = {}
phase_responses = {}
for n_tanks in [1, 2, 3, 4]:  # number of tanks in series
    tau_h = freq_times_tau_h / omega  # total residence time (min)
    amplitude_responses[n_tanks] = calc_amplitude_response(
        omega, tau_h, n_tanks
    )
    phase_responses[n_tanks] = calc_phase_response(omega, tau_h, n_tanks)

plt.figure(figsize=FIGSIZE)
for n_tanks, amplitude_response in amplitude_responses.items():
    plt.loglog(freq_times_tau_h, amplitude_response, label=n_tanks)

plt.xlim([1e-1, 1e2])
plt.ylim([1e-3, 1.2])
plt.xlabel(r"Frequency $\times$ total residence time")
plt.ylabel("Amplitude Ratio")
plt.grid()
# plt.legend(title='$n$')
plt.annotate("n=1", xy=(48, 0.03), color="C0")
plt.annotate("n=2", xy=(29, 0.007), color="C1")
plt.annotate("n=3", xy=(20, 0.0035), color="C2")
plt.annotate("n=4", xy=(11, 0.002), color="C3")
plt.title("Frequency Responses of Buffer Tanks in Series")
plt.tight_layout()
plt.savefig(plot_dir / "buffer_tanks_amplitude_response.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=FIGSIZE)
for i, (n_tanks, phase_response) in enumerate(phase_responses.items()):
    plt.plot(freq_times_tau_h, phase_response * 180 / np.pi, label=n_tanks)
    x, y = freq_times_tau_h[-1] * 0.9, phase_response[-1] * 180 / np.pi + 8
    plt.annotate(f"n={n_tanks}", xy=(x, y), color=f"C{i}", ha="right")

plt.xscale("log")
plt.xlim([1e-1, 1e2])
plt.yticks(np.linspace(-360, 0, 9))
plt.ylim([-360, 0])
plt.xlabel(r"Frequency $\times$ total residence time")
plt.ylabel("Phase Shift (degrees)")
plt.grid()
plt.title("Phase Shift of Buffer Tanks in Series")
plt.tight_layout()
plt.savefig(plot_dir / "buffer_tanks_phase_shift.png", dpi=300)
plt.show()

## Mixed Tank Example

In [ ]:
# System parameters
total_volume = np.logspace(0, 3, 101)  # total volume of buffer tanks (m^3)
flow_rate = 1  # m^3 / min

# Input disturbance parameters
period = 60  # period of inlet disturbance (min)
omega = 2 * np.pi / period  # angular frequency (rad/min)

amplitude_responses = {}
for n_tanks in [1, 2, 3]:  # number of tanks in series
    tau_h = total_volume / flow_rate  # total residence time (min)
    amplitude_responses[n_tanks] = calc_amplitude_response(
        omega, tau_h, n_tanks
    )

plt.figure(figsize=FIGSIZE)
for n_tanks, amplitude_response in amplitude_responses.items():
    plt.loglog(total_volume, amplitude_response, label=n_tanks)

plt.xlim([1, 1e3])
plt.ylim([1e-2, 1.2])
plt.xlabel(r"Total tank volume (m$^3$)")
plt.ylabel("Amplitude Ratio")
plt.grid()
plt.legend(title="$n$")
# plt.title(
#     f"Example: Effect of Tank Volume on Attenuation"
# )
plt.tight_layout()
plt.savefig(plot_dir / "buffer_tanks_ex_var_volume.png", dpi=300)
plt.show()

In [ ]:
# Input disturbance parameters
x_lim = (1e-2, 10)  # frequency range (1/hour)
frequency = np.logspace(np.log10(x_lim[0]), np.log10(x_lim[1]), 101)
omega = 2 * np.pi * frequency / 60  # angular frequency (rad/min)
total_volume = 90  # total volume of buffer tanks (m^3)

amplitude_responses = {}
for n_tanks in [1, 2, 3, 4]:  # number of tanks in series
    tau_h = total_volume / flow_rate  # total residence time (min)
    amplitude_responses[n_tanks] = calc_amplitude_response(
        omega, tau_h, n_tanks
    )

plt.figure(figsize=FIGSIZE)
for n_tanks, amplitude_response in amplitude_responses.items():
    plt.loglog(frequency, amplitude_response, label=n_tanks)

plt.xlim(x_lim)
plt.ylim([1e-2, 1.2])
plt.xlabel(r"Frequency [$hour^{-1}$]")
plt.ylabel("Amplitude Ratio")
plt.grid()
plt.legend(title="$n$")
plt.title(
    rf"Example: Frequency Responses with Total Tank Capacity of {total_volume} $m^3$"
)
plt.tight_layout()
plt.savefig(plot_dir / "buffer_tanks_ex_var_freq.png", dpi=300)
plt.show()

## Generate Bounded Random Walk (BRW) Sequences

In [ ]:
seed = 100
rng = np.random.default_rng(seed)

# Noise std. dev.
sd_e = 5.0

# Bounded random walk parameters
r1 = -40.0  # When x = r1, bias = +1 (pushes up)
r2 = 40.0  # When x = r2, bias = -1 (pushes down)
a1 = 0.2  # aggressiveness of lower bound
a2 = 0.2  # aggressiveness of upper bound

# Number of random walks to generate
n_walks = 3

nT = 600
bounded_random_walks = sample_bounded_random_walk(sd_e, r1, r2, a1, a2, nT, n_walks=3, rng=rng)
assert bounded_random_walks.shape == (nT, n_walks)

# Nominal input value
u_nop = 50.0

# Time vector
Ts = 1.0
t = Ts * np.arange(nT)

In [ ]:
# Only plot the first t_stop minutes of each BRW
t_stop = 600.0
nT_plot = int(np.floor(t_stop / Ts))

n_plots = min(5, n_walks)

marker = ""

fig, axes = plt.subplots(n_plots, 1, sharex=True, figsize=(7, 1 + 1.5*n_plots))

for i, ax in enumerate(axes):
    u = bounded_random_walks[:, i]
    ax.plot(t[:nT_plot], u_nop + u[:nT_plot], marker=marker)
    ax.axhline(u_nop + r1, linestyle='--', color='grey', label='min')
    ax.axhline(u_nop + r2, linestyle='--', color='grey', label='max')
    ax.set_ylim([u_nop + 1.3 * r1, u_nop + 1.3 * r2])
    ax.set_ylabel("%")
    ax.grid()
    ax.set_title(f"Bounded Random Walk {i+1:d}")

ax.set_xlabel("Time (mins)")
plt.tight_layout()
filename = "bounded_random_walks.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()

## Mixed Tank Simulation

In [ ]:
# Tank dimensions
H = 9   # Height [m]
A = 10  # Cross-sectional area [m^2]

# System parameters
tank_volume = A * H  # m^3
flow_rate = 1  # m^3 / min

flow_rate_nop = 1  # m^3 / min
residence_time = tank_volume / flow_rate_nop  # mins
residence_time  # mins

In [ ]:
# Transfer function of system
tau = residence_time
G = con.tf([1], [tau, 1])
print(G)

In [ ]:
# Step response
t_out, y_out = con.step_response(G, T=t)

# Plot
fig, ax = plt.subplots()
ax.plot(t_out, y_out)
ax.set_xlabel("Time (mins)")
ax.set_ylabel("Output")
ax.set_title("Step Response")
ax.grid(True)
plt.show()

In [ ]:
# Step response                                                                                                                      
t_out, y_out = con.step_response(G, T=t)                                                                                                              

# Input (unit step)                                                                                                                                   
u = np.ones_like(t_out)                                                                                                                             
u[0] = 0  # step starts at t=0, show from zero

# 63.2% of steady-state value at t = tau
y_ss = y_out[-1]
y_tau = 0.632 * y_ss

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 5))

# Upper: output
ax = axes[0]
ax.plot(t_out, y_out)
ax.axvline(tau, color="gray", linestyle="--", linewidth=1)
ax.axhline(y_tau, color="gray", linestyle="--", linewidth=1)
ax.plot(tau, y_tau, "ko", markersize=4)
ax.annotate(
    f"response at $\\tau$ = {tau:.0f} min",
    xy=(tau, y_tau),
    xytext=(tau + 10, y_tau - 0.15),
    arrowprops=dict(arrowstyle="->", color="black"),
)
ax.set_ylabel("Output, $y(k)$")
ax.set_title("Step Response")
ax.grid(True)

# Lower: input
ax = axes[1]
ax.plot(t_out, u)
ax.set_xlabel("Time (mins)")
ax.set_ylabel("Input, $u(k)$")
ax.set_ylim(-0.1, 1.5)
ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Pick one of the input sequences
u = bounded_random_walks[:, 1]

t_out, yout = con.forced_response(G, T=t, U=u, X0=0)


marker = ""

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(t[:nT_plot], u_nop + u[:nT_plot], marker=marker, label='in')
ax.plot(t[:nT_plot], (yout + u_nop)[:nT_plot], marker=marker, label='out')
ax.axhline(u_nop + r1, linestyle='--', color='grey', label='min')
ax.axhline(u_nop + r2, linestyle='--', color='grey', label='max')
ax.set_ylim([u_nop + 1.3 * r1, u_nop + 1.3 * r2])
ax.set_xlabel("Time (mins)")
ax.set_ylabel("Composition (%)")
ax.grid()
ax.legend()
#ax.set_title("Mixed Tank - Compositions")
plt.tight_layout()
filename = "mixing_tank_sim.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()


In [ ]:
# Simulating n tanks in series
n_tanks = 2

flow_rate = 1  # m^3 / min
flow_rate_nop = 1  # m^3 / min

tfs = {}
for n_tanks in [1, 2, 3]:

    # System parameters
    tank_volume = A * H / n_tanks  # m^3
    residence_time = tank_volume / flow_rate_nop  # mins

    # Transfer function of system
    tau = residence_time
    G = con.tf([1], [tau, 1]) ** n_tanks
    print(G.den[0][0])
    tfs[n_tanks] = G


In [ ]:
# Pick one of the input sequences
u = bounded_random_walks[:, 1]

marker = ""

fig, ax = plt.subplots(figsize=(7, 2.5))

ax.plot(t[:nT_plot], u_nop + u[:nT_plot], marker=marker, label='in')
for n_tanks, G in tfs.items():
    t_out, yout = con.forced_response(G, T=t, U=u, X0=0)
    ax.plot(t[:nT_plot], (yout + u_nop)[:nT_plot], marker=marker, label=f"out {n_tanks}")
ax.set_ylim([u_nop + 1.3 * r1, u_nop + 1.3 * r2])
ax.set_xlabel("Time (mins)")
ax.set_ylabel("Composition (%)")
ax.grid()
ax.legend()
#ax.set_title("Mixed Tank - Compositions")
plt.tight_layout()
filename = "mixing_tank_sim_n_tanks.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()

## Smoothing Flow Rate Disturbances

For flow rate disturbances, the objective of the buffer tank is to smoothen the flow response, $q(s) = h(s) q_{in}(s)$, so that the variations in $q$ are smaller than in $q_{in}$.

Let $k(s)$ denote the transfer function for the level controller including measurement and actuator dynamics, and the inner flow control loop. 

Then $q(s) = k(s)(V(s) - V_s)$ where $V_s$ is the set-point for the volume. Combining this with the total mass balance from above yields

$$
q(s) = \frac{k(s)}{s + k(s)} q_{in}(s) - \frac{sk(s)}{s + k(s)} V_s
$$

The buffer tank transfer function is thus given by

$$
h(s) = \frac{k(s)}{s + k(s)} = \frac{1}{\frac{s}{k(s)} + 1}
$$

With a proportional controller $k(s) = K$, $h(s)$ is a first order filter with $\tau = 1/K$.  For a given $h(s)$, the controller is

$$
k(s) = \frac{s h(s)}{1 - h(s)}
$$

In [ ]:
# Check derivation
s, K, tau = sympy.symbols("s, K, tau")

h = 1 / (s / K + 1)

sympy.simplify(s * h / (1 - h))

In [ ]:
# Try second order smoothing
h = 1 / (tau * s + 1) ** 2

sympy.simplify(s * h / (1 - h))

## Surge Tank Simulation

In [ ]:
# Tank dimensions
H = 9   # Height [m]
A = 10  # Cross-sectional area [m^2]

# System parameters
tank_volume = A * H  # m^3

q_min, q_max = 0.0, 2.0  # m^3 / min
q_nop = 1.0  # m^3 / min

# Level set point
L_sp = H / 2

# level controller: proportional gain
K = max(q_max - q_nop, q_nop - q_min) / A / H * 2

# Closed loop time constant
tau = 1 / K

G = con.tf([1], [tau, 1])

print(f"{A = }\n{K = }\n{tau = }\n{G}")

In [ ]:
scale_factor = max(q_max - q_nop, q_nop - q_min) / 50.0

# Pick one of the input sequences
u = bounded_random_walks[:, 1] * scale_factor
t_out, yout = con.forced_response(G, T=t, U=u, X0=0)

y_nop = q_nop
q_in = q_nop + u
q_out = q_nop + yout

level = L_sp + np.cumsum((q_in - q_out) / A)

marker = ""

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(t[:nT_plot], q_in[:nT_plot], marker=marker, label='in')
ax.plot(t[:nT_plot], q_out[:nT_plot], marker=marker, label='out')
ax.axhline(q_nop + r1 * scale_factor, linestyle='--', color='grey', label='min')
ax.axhline(q_nop + r2 * scale_factor, linestyle='--', color='grey', label='max')
ax.set_xlabel("Time (mins)")
ax.set_ylabel("Flow rate ($m^3/min$)")
ax.grid()
ax.legend()
#ax.set_title("Surge Tank - Flow Rates")
plt.tight_layout()
filename = "surge_tank_sim.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(t[:nT_plot], level[:nT_plot], marker=marker, color='C2', label='level')
ax.axhline(0, linestyle='--', color='grey', label='min')
ax.axhline(H, linestyle='--', color='grey', label='max')
ax.set_xlabel("Time (mins)")
ax.set_ylabel("Level (m)")
ax.grid()
ax.legend()
#ax.set_title("Surge Tank - Level")
plt.tight_layout()
filename = "surge_tank_level.png"
plt.savefig(plot_dir / filename, dpi=300)
plt.show()

In [ ]:
# Double BRW



## Examples from Seborg

**Example 4.7**

Two surge tanks are placed in series so that the exit flow from the top tank flows into the lower tank, as shown in Fig. 4.4. If an exit flow rate is proportional to liquid level (or head) in that tank, derive the transfer function that relates changes in the exit flow rate $q_2$ of the lower tank to changes in the inlet flow rate to the top tank $q_i$. Show how this overall transfer function, $Q'_2(s)/Q'_i(s)$, is related to the individual transfer functions, $H'_2(s)/Q'_i(s)$, $Q'_1(s)/H'_1(s)$, $Q'_1(s)/H'_1(s)$, $H'_2(s)/Q'_1(s)$, and $Q'_2(s)/H'_2(s)$.

$H'_1(s)$ and $H'_2(s)$ denote the Laplace transforms of the deviations in Tank 1 and Tank 2 levels, respectively. Assume that the two tanks have cross-sectional areas, $A_1$ and $A_2$, and valve resistances, $R_1$ and $R_2$, respectively.

**Figure 4.4 from Seborg**

<img src="images/seborg_fig4.4.png" width="40%">

In [ ]:
t, s = sympy.symbols("t, s")

A1, R1, A2, R2 = sympy.symbols("A1, R1, A2, R2")

# Define h1(t) as a function of time
h1 = sympy.Function("h1")
qi = sympy.Function("qi")

# Assume linear relationship between tank level and flow
# q1 = h1 / R1
# A1 * d h1 / dt = q1 - qi

ode1 = sympy.Eq(A1 * h1(t).diff(t) - h1(t) / R1 + qi(t), 0)
ode1

In [ ]:
ode1.lhs

In [ ]:
#laplace_transform(ode1.lhs, t, s, noconds=True)

In [ ]:
# Take Laplace transform (assuming zero initial conditions)
from sympy.integrals.transforms import laplace_transform

laplace_transform(h1(t).diff(t), t, s, noconds=True)